# Head x MLP additivity — instruct models (Appendix N, Table 22)

Reproduces **Table 22** of the paper: additivity of the two causal components
on the directional endpoint **P(R) match**:

* **A** = edge-KO of the binding heads only (R→item, `B_to_item`)
* **B** = MLP mean-ablation of the top-K POS neurons only
* **C** = both, stacked in the same forward
* interaction = `C − (A+B)`, item-level paired test (n=66), plus the ratio `C/(A+B)`

`C ≈ A+B` → parallel pathways; `C < A+B` → shared/series (read→write).
Only the three pre-norm models with a localizable MLP component apply
(Mistral K=40, Llama K=20, Nemo K=20); Gemma has no component to compose;
the softcapping assert below enforces this.

**Input**: `./results/<MODEL_KEY>_instruct/results_<MODEL_KEY>_instruct_mlp_neuron_filtered.pkl`
(written by the DLA neurons notebook), loaded, never recomputed, never mutated.
**Output**: `./results/<MODEL_KEY>_instruct/results_<MODEL_KEY>_additivity.pkl`.


In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "nemo"}, no gemma2 (softcapping assert below)
KNEE = {"mistral":40, "llama":20, "nemo":20}
K_ADD = None           # knee per model: Mistral 40, Llama 20, Nemo 20
if K_ADD is None: K_ADD = KNEE[MODEL_KEY]
print("model:", MODEL_KEY, "| K:", K_ADD)


In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
HF_TOKEN = config.HF_TOKEN          # read from the HF_TOKEN env var
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
ACTIVE_MODEL = config.ACTIVE_MODEL

# Head-side helper functions, imported from common/.
from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import format_for_chat, find_option_token_ids, detect_spans
from common.instruct.hooks import edge_knockout, compute_logit_scores_edge


In [ ]:
import os, json, pickle, re
import numpy as np, torch
import torch.nn.functional as F
from pathlib import Path
from contextlib import contextmanager, ExitStack
from scipy.stats import ttest_1samp, ttest_rel


In [ ]:
import os
import re
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager
from itertools import combinations

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy import stats
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)


In [ ]:
print("=" * 80)
print(f"STAGE 1: S-SCORES — {CFG['label']}")
print("=" * 80)

# Load data
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# Load model
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
config.model = model; config.tokenizer = tokenizer

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
config.first_device = first_device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# Format texts
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# Compute S-scores
print(f"\n  Computing S-scores...")
results_sscore = {}
for cond in conditions:
    scores, c_chosen, p_c_list = [], [], []
    for i, text in enumerate(texts_fmt[cond]):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
        scores.append(S)
        chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
        c_chosen.append(1 if chosen == 'c' else 0)
        lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
        p_c_list.append(np.exp(lp_c - lp_all))
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            print(f"    {cond}: {i}/{n_total}")
            torch.cuda.empty_cache()
    results_sscore[cond] = {
        'S': np.array(scores), 'c_chosen': np.array(c_chosen),
        'p_c': np.array(p_c_list),
    }

# Print results
print(f"\n{'':20s}  {'Mean S':>8s}  {'Med S':>8s}  {'(c) rate':>8s}  {'Mean P(c)':>9s}")
for cond in conditions:
    r = results_sscore[cond]
    print(f"  {cond:20s}  {r['S'].mean():8.3f}  {np.median(r['S']):8.3f}  "
          f"{r['c_chosen'].mean():8.3f}  {r['p_c'].mean():9.3f}")
delta_S = results_sscore['B_cult']['S'].mean() - results_sscore['B_unrel']['S'].mean()
print(f"\n  Δ(S) = {delta_S:.4f}")


In [ ]:
assert not CFG["has_softcapping"], "this additivity cell targets the 3 pre-norm models"
FINAL_HEADS = CFG["heads"]
assoc_pos_arr = list(data["assoc_pos"])
n_layers = model.config.num_hidden_layers
first_device = next(model.parameters()).device
print("  model loaded:", CFG["label"], "| heads:", FINAL_HEADS)


In [ ]:
# load neuron_store from the saved pickle (no recompute)
_npkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_mlp_neuron_filtered.pkl"
with open(_npkl, "rb") as f:
    neuron_store = pickle.load(f)
kept = neuron_store["kept"]                         # (l, j, Δdir, leak), ranked by ratio
pos_pairs = [(l, j) for (l, j, d, lk) in kept if d > 0][:K_ADD]
POS = {}
for l, j in pos_pairs: POS.setdefault(l, []).append(j)
print(f"  loaded {_npkl.name}")
print(f"  MLP POS set: {sum(len(v) for v in POS.values())} neurons at K={K_ADD}")


In [ ]:
# MLP mean-ablation context manager (self-contained)
print("  computing mean neuron activations (50+50 baseline)...")
d_ff = model.config.intermediate_size
_sum = torch.zeros(n_layers, d_ff, dtype=torch.float64, device=first_device); _nt = 0
def _mk(l):
    def hook(m, args): _sum[l] += args[0][0].detach().to(torch.float64).sum(0)
    return hook
_mh = [model.model.layers[l].mlp.down_proj.register_forward_pre_hook(_mk(l)) for l in range(n_layers)]
try:
    with torch.no_grad():
        for cond in conditions:
            for i in range(50):
                enc = tokenizer(texts_fmt[cond][i], return_tensors="pt", truncation=True, max_length=512)
                enc = {k: v.to(first_device) for k, v in enc.items()}
                _ = model(**enc); _nt += enc["input_ids"].shape[1]; del enc, _
finally:
    for h in _mh: h.remove()
neuron_mean_acts = (_sum / _nt).to(model.dtype)

@contextmanager
def ablate(nd):
    H = []
    def mk(l, js):
        jt = torch.tensor(js, device=first_device)
        def hook(m, args):
            x = args[0].clone(); x[..., jt] = neuron_mean_acts[l, jt]; return (x,) + args[1:]
        return hook
    try:
        for l, js in nd.items():
            H.append(model.model.layers[l].mlp.down_proj.register_forward_pre_hook(mk(l, list(js))))
        yield
    finally:
        for h in H: h.remove()
print("  ablation ctx ready")


In [ ]:
# build per-prompt positions on B_cult (R = associated = B_tokens)
print("  building positions...")
positions, valid = [], 0
for i in range(len(texts_fmt["B_cult"])):
    q_text = data["B_cult"][i].split("\n\n")[0]
    oa, ob = extract_options(q_text)
    item = data["items_cult"][i]
    enc = tokenizer(texts_fmt["B_cult"][i], return_tensors="pt")
    ids = enc["input_ids"][0].tolist()
    sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
    if sp is None:
        positions.append(None)
    else:
        ap = assoc_pos_arr[i]
        B_tokens = sp["opt_a"] if ap == "a" else sp["opt_b"]   # R
        A_tokens = sp["opt_b"] if ap == "a" else sp["opt_a"]   # U
        positions.append({"item_tokens": sp["item"], "B_tokens": B_tokens, "A_tokens": A_tokens})
        valid += 1
    del enc
print(f"  valid positions: {valid}/{len(positions)}")
items_arr = np.array(data["items_cult"])

def pR_match(use_heads, use_neurons):
    out = np.full(len(texts_fmt["B_cult"]), np.nan)
    for i in range(len(texts_fmt["B_cult"])):
        pos = positions[i]
        if pos is None: continue
        enc = tokenizer(texts_fmt["B_cult"][i], return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with ExitStack() as stack:
            if use_heads:
                stack.enter_context(edge_knockout(model, FINAL_HEADS, pos["item_tokens"], pos["B_tokens"]))
            if use_neurons:
                stack.enter_context(ablate(use_neurons))
            with torch.no_grad():
                o = model(**enc)
        lp = F.log_softmax(o.logits[0, -1, :].float(), dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens["a"]], 0).item()
        lp_b = torch.logsumexp(lp[option_tokens["b"]], 0).item()
        ap = assoc_pos_arr[i]
        lp_R, lp_U = (lp_a, lp_b) if ap == "a" else (lp_b, lp_a)
        out[i] = 1.0 / (1.0 + np.exp(-(lp_R - lp_U)))
        del enc, o, lp
        if i % 200 == 0 and i > 0: torch.cuda.empty_cache()
    return out


In [ ]:
# measure baseline, A, B, C and the interaction
print("  baseline...");          p0 = pR_match(False, None)
print("  A heads only...");      pA = pR_match(True,  None)
print("  B MLP only...");        pB = pR_match(False, POS)
print("  C both...");            pC = pR_match(True,  POS)

im = lambda v: np.array([np.nanmean(v[items_arr == it]) for it in np.unique(items_arr)])
P0, A, B, C = im(p0), im(pA), im(pB), im(pC)
dA, dB, dC = A - P0, B - P0, C - P0
inter = dC - (dA + dB)
tI, pI = ttest_1samp(inter, 0)

print("\n" + "=" * 70)
print(f"  ADDITIVITY  QK heads x MLP neurons — {CFG['label']}  (P(R) match, n={len(P0)})")
print("=" * 70)
print(f"  baseline P(R)          {P0.mean():.4f}")
print(f"  A heads only     ΔP(R) {dA.mean():+.4f}    (cf. tab:directional-ko Δ_R)")
print(f"  B MLP only       ΔP(R) {dB.mean():+.4f}")
print(f"  A+B  (parallel)        {(dA+dB).mean():+.4f}")
print(f"  C both           ΔP(R) {dC.mean():+.4f}")
print(f"  interaction C-(A+B)    {inter.mean():+.4f}  (t={tI:+.2f}, p={pI:.2e})")
ratio = dC.mean()/(dA+dB).mean() if (dA+dB).mean()!=0 else float('nan')
verdict = ("sub-additive (shared/series, read->write)" if inter.mean()>1e-4
           else "super-additive" if inter.mean()<-1e-4 else "additive (parallel)")
print(f"\n  C/(A+B) = {ratio:.2f}  ->  {verdict}")

# The additivity dict is written to its own artifact,
# so the input neuron pickle loaded above
# is never mutated.
additivity = {"P0":float(P0.mean()),"A":float(dA.mean()),"B":float(dB.mean()),
    "C":float(dC.mean()),"inter":float(inter.mean()),"t":float(tI),"p":float(pI),"K":K_ADD}
_apkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_additivity.pkl"
with open(_apkl, "wb") as f: pickle.dump(additivity, f)
print(f"\n  saved additivity to {_apkl.name}")


## Reading

* **Check A first**: it should ≈ the `Δ_R` of `tab:directional-ko` for this model
  (Mistral −0.082, Llama −0.041, Nemo −0.032). If A matches, the head-side
  knockout is correctly set up and B/C are trustworthy.
* **interaction > 0** (C above A+B, i.e. smaller combined effect) → heads and
  neurons **share** part of the pathway (read feeds write).
* **interaction ≈ 0** → parallel, independent contributions.
* Run the three pre-norm models (set `MODEL_KEY`, restart, Run all).
